## Simple pyramid + metrics

### Pyramid

In [ ]:
import cv2
import numpy as np
import math
from ultralytics import YOLO
from typing import List, Tuple, Optional, Dict
import hashlib
import os
from pathlib import Path
from tqdm import tqdm
import json

class SmartYOLOInference:
    def __init__(self, model_path: str, imgsz: int = 640, conf_thres: float = 0.25, iou_thres: float = 0.5):
        self.model = YOLO(model_path)
        self.imgsz = imgsz
        self.conf_thres = conf_thres
        self.iou_thres = iou_thres
        self.tile_size = imgsz
        self.color_cache = {}
        self.class_names = ['ship', 'plane', 'helicopter', 'buoy', 'drone', 'person']

    def get_class_color(self, class_id: int) -> Tuple[int, int, int]:
        if class_id in self.color_cache:
            return self.color_cache[class_id]
        
        hash_obj = hashlib.md5(str(class_id).encode())
        hash_int = int(hash_obj.hexdigest(), 16)
        
        h = hash_int % 360
        s = 0.7 + (hash_int % 100) / 500
        l = 0.5 + (hash_int % 100) / 500
        
        c = (1 - abs(2 * l - 1)) * s
        x = c * (1 - abs((h / 60) % 2 - 1))
        m = l - c / 2
        
        if h < 60:
            r, g, b = c, x, 0
        elif h < 120:
            r, g, b = x, c, 0
        elif h < 180:
            r, g, b = 0, c, x
        elif h < 240:
            r, g, b = 0, x, c
        elif h < 300:
            r, g, b = x, 0, c
        else:
            r, g, b = c, 0, x
        
        color = (int((b + m) * 255), int((g + m) * 255), int((r + m) * 255))
        self.color_cache[class_id] = color
        return color

    def calculate_optimal_stride(self, dimension: int, tile_size: int, min_overlap: float = 0.2, max_overlap: float = 0.7) -> Tuple[int, float]:
        min_tiles = math.ceil(dimension / tile_size)
        best_stride = tile_size
        best_overlap = 0.0
        
        for n_tiles in range(min_tiles, min_tiles + 10): 
            if n_tiles == 1:
                stride = dimension
            else:
                stride = (dimension - tile_size) / (n_tiles - 1)
            
            if stride <= 0: 
                continue
            
            overlap = 1.0 - (stride / tile_size)
            
            if min_overlap <= overlap <= max_overlap:
                if stride > best_stride:
                    best_stride = int(stride)
                    best_overlap = overlap
        
        if best_overlap == 0.0:
            best_overlap = 0.3
            best_stride = int(tile_size * (1 - best_overlap))
            
        return best_stride, best_overlap

    def slice_image(self, image: np.ndarray, tile_size: int) -> List[dict]:
        h, w = image.shape[:2]
        stride_x, overlap_x = self.calculate_optimal_stride(w, tile_size)
        stride_y, overlap_y = self.calculate_optimal_stride(h, tile_size)
        
        slices = []
        for y in range(0, h, stride_y):
            for x in range(0, w, stride_x):
                y_end = min(y + tile_size, h)
                x_end = min(x + tile_size, w)
                
                crop = image[y:y_end, x:x_end]
                ch, cw = crop.shape[:2]
                
                if ch < tile_size or cw < tile_size:
                    padded_crop = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                    padded_crop[:ch, :cw] = crop
                    crop = padded_crop
                
                slices.append({
                    'crop': crop,
                    'offset': (x, y),
                    'orig_size': (cw, ch),
                    'tile_size': tile_size
                })
        return slices

    def run_tiled_inference(self, image: np.ndarray, tile_size: int) -> List[np.ndarray]:
        h, w = image.shape[:2]
        
        if tile_size >= min(w, h):
            results = self.model(image, conf=self.conf_thres, iou=self.iou_thres, verbose=False)
            all_boxes = []
            
            if results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                confs = results[0].boxes.conf.cpu().numpy()
                classes = results[0].boxes.cls.cpu().numpy()
                
                for j in range(len(boxes)):
                    all_boxes.append([
                        boxes[j][0], boxes[j][1], boxes[j][2], boxes[j][3],
                        confs[j], classes[j],
                        'pyramid', tile_size
                    ])
            return all_boxes
        
        slices = self.slice_image(image, tile_size)
        all_boxes = []
        
        batch_size = 8
        for i in range(0, len(slices), batch_size):
            batch_slices = slices[i:i+batch_size]
            crops = [s['crop'] for s in batch_slices]
            
            results = self.model(crops, conf=self.conf_thres, iou=self.iou_thres, verbose=False)
            
            for res, slice_info in zip(results, batch_slices):
                if res.boxes is None:
                    continue
                
                boxes = res.boxes.xyxy.cpu().numpy()
                confs = res.boxes.conf.cpu().numpy()
                classes = res.boxes.cls.cpu().numpy()
                
                offset_x, offset_y = slice_info['offset']
                
                for j in range(len(boxes)):
                    x1, y1, x2, y2 = boxes[j]
                    
                    scale_x = slice_info['orig_size'][0] / tile_size
                    scale_y = slice_info['orig_size'][1] / tile_size
                    
                    global_x1 = max(0, min(x1 * scale_x + offset_x, w))
                    global_y1 = max(0, min(y1 * scale_y + offset_y, h))
                    global_x2 = max(0, min(x2 * scale_x + offset_x, w))
                    global_y2 = max(0, min(y2 * scale_y + offset_y, h))
                    
                    all_boxes.append([
                        global_x1, global_y1, global_x2, global_y2, 
                        confs[j], classes[j],
                        'tiled', tile_size
                    ])
        
        return all_boxes

    def run_global_inference(self, image: np.ndarray) -> List[np.ndarray]:
        results = self.model(image, conf=self.conf_thres, iou=self.iou_thres, verbose=False)
        all_boxes = []
        
        if results[0].boxes is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            confs = results[0].boxes.conf.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()
            
            for j in range(len(boxes)):
                all_boxes.append([
                    boxes[j][0], boxes[j][1], boxes[j][2], boxes[j][3],
                    confs[j], classes[j],
                    'global', 0
                ])
        
        return all_boxes

    def calculate_iou(self, box1: np.ndarray, box2: np.ndarray) -> float:
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        union_area = box1_area + box2_area - inter_area
        if union_area == 0: 
            return 0
        return inter_area / union_area

    def calculate_containment(self, inner_box: np.ndarray, outer_box: np.ndarray) -> float:
        x1 = max(inner_box[0], outer_box[0])
        y1 = max(inner_box[1], outer_box[1])
        x2 = min(inner_box[2], outer_box[2])
        y2 = min(inner_box[3], outer_box[3])
        
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        inner_area = (inner_box[2] - inner_box[0]) * (inner_box[3] - inner_box[1])
        
        if inner_area == 0:
            return 0
        return inter_area / inner_area

    def merge_same_class_boxes(self, boxes: List[np.ndarray], merge_iou_thres: float = 0.5) -> List[np.ndarray]:
        if not boxes:
            return boxes
        
        class_groups = {}
        for box in boxes:
            cls_id = int(box[5])
            if cls_id not in class_groups:
                class_groups[cls_id] = []
            class_groups[cls_id].append(box)
        
        merged_boxes = []
        
        for cls_id, class_boxes in class_groups.items():
            merged = []
            used = [False] * len(class_boxes)
            
            for i in range(len(class_boxes)):
                if used[i]:
                    continue
                
                current_box = class_boxes[i].copy()
                used[i] = True
                
                for j in range(i + 1, len(class_boxes)):
                    if not used[j]:
                        iou = self.calculate_iou(current_box, class_boxes[j])
                        if iou > merge_iou_thres:
                            current_box[0] = min(current_box[0], class_boxes[j][0])
                            current_box[1] = min(current_box[1], class_boxes[j][1])
                            current_box[2] = max(current_box[2], class_boxes[j][2])
                            current_box[3] = max(current_box[3], class_boxes[j][3])
                            current_box[4] = max(current_box[4], class_boxes[j][4])
                            if class_boxes[j][6] == 'global':
                                current_box[6] = 'global'
                                current_box[7] = 0
                            used[j] = True
                
                merged.append(current_box)
            
            merged_boxes.extend(merged)
        
        return merged_boxes

    def fuse_pyramid_levels(self, pyramid_boxes: Dict[int, List[np.ndarray]], global_boxes: List[np.ndarray],
                           nms_iou_thres: float = 0.5, containment_thres: float = 0.8) -> List[np.ndarray]:
        all_pyramid_boxes = []
        for tile_size, boxes in pyramid_boxes.items():
            all_pyramid_boxes.extend(boxes)
        
        final_boxes = []
        for box in global_boxes:
            final_boxes.append(box)
        
        for pyramid_box in all_pyramid_boxes:
            is_duplicate = False
            
            for global_box in final_boxes:
                if int(pyramid_box[5]) != int(global_box[5]):
                    continue
                
                iou = self.calculate_iou(pyramid_box, global_box)
                if iou > nms_iou_thres:
                    is_duplicate = True
                    break
                
                containment_pyramid_in_global = self.calculate_containment(pyramid_box, global_box)
                if containment_pyramid_in_global > containment_thres:
                    is_duplicate = True
                    break
                
                containment_global_in_pyramid = self.calculate_containment(global_box, pyramid_box)
                if containment_global_in_pyramid > containment_thres:
                    is_duplicate = True
                    break
            
            if not is_duplicate:
                final_boxes.append(pyramid_box)
        
        return final_boxes

    def boxes_to_yolo_format(self, boxes: List[np.ndarray], image_width: int, image_height: int) -> List[str]:
        yolo_lines = []
        
        for box in boxes:
            x1, y1, x2, y2, conf, cls_id = box[:6]
            
            x_center = (x1 + x2) / 2.0
            y_center = (y1 + y2) / 2.0
            width = x2 - x1
            height = y2 - y1
            
            x_center_norm = x_center / image_width
            y_center_norm = y_center / image_height
            width_norm = width / image_width
            height_norm = height / image_height
            
            yolo_line = f"{int(cls_id)} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}"
            yolo_lines.append(yolo_line)
        
        return yolo_lines

    def save_detections_to_txt(self, boxes: List[np.ndarray], image_width: int, image_height: int, 
                               txt_path: str, include_conf: bool = False):
        Path(txt_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(txt_path, 'w', encoding='utf-8') as f:
            for box in boxes:
                x1, y1, x2, y2, conf, cls_id = box[:6]
                
                x_center = (x1 + x2) / 2.0
                y_center = (y1 + y2) / 2.0
                width = x2 - x1
                height = y2 - y1
                
                x_center_norm = x_center / image_width
                y_center_norm = y_center / image_height
                width_norm = width / image_width
                height_norm = height / image_height
                
                if include_conf:
                    line = f"{int(cls_id)} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f} {conf:.6f}\n"
                else:
                    line = f"{int(cls_id)} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}\n"
                
                f.write(line)

    def _process_image_array(self, image: np.ndarray, pyramid_levels: List[int] = None,
                            merge_iou_thres: float = 0.5, fuse_nms_thres: float = 0.5,
                            containment_thres: float = 0.8, verbose: bool = True):
        h, w = image.shape[:2]
        
        if pyramid_levels is None:
            pyramid_levels = [self.imgsz]
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Размер изображения: {w}x{h}")
            print(f"Уровни пирамиды: {pyramid_levels}")
            print(f"{'='*60}")
        
        pyramid_boxes = {}
        for tile_size in pyramid_levels:
            if verbose:
                print(f"\n[Level] Tile size: {tile_size}x{tile_size}")
            
            tiled_boxes = self.run_tiled_inference(image, tile_size)
            tiled_boxes_merged = self.merge_same_class_boxes(tiled_boxes, merge_iou_thres)
            pyramid_boxes[tile_size] = tiled_boxes_merged
            
            if verbose:
                print(f"  Найдено: {len(tiled_boxes)} → После слияния: {len(tiled_boxes_merged)}")
        
        if verbose:
            print(f"\n[Global] 4K оригинал...")
        global_boxes = self.run_global_inference(image)
        if verbose:
            print(f"  Найдено: {len(global_boxes)}")
        
        if verbose:
            print(f"\n[Fusion] Слияние уровней с приоритетом 4K...")
        final_boxes = self.fuse_pyramid_levels(
            pyramid_boxes, 
            global_boxes,
            nms_iou_thres=fuse_nms_thres,
            containment_thres=containment_thres
        )
        if verbose:
            print(f"  Итого: {len(final_boxes)} объектов")
        
        global_count = sum(1 for b in final_boxes if len(b) > 6 and b[6] == 'global')
        tiled_count = len(final_boxes) - global_count
        
        stats = {
            'image_size': (w, h),
            'pyramid_levels': pyramid_levels,
            'boxes_per_level': {k: len(v) for k, v in pyramid_boxes.items()},
            'global_boxes': len(global_boxes),
            'final_boxes': len(final_boxes),
            'global_in_final': global_count,
            'tiled_in_final': tiled_count,
        }
        
        return final_boxes, stats

    def process_image(self, image_path: str, save_path: Optional[str] = None,
                     txt_save_dir: Optional[str] = None,
                     pyramid_levels: List[int] = None,
                     merge_iou_thres: float = 0.5,
                     fuse_nms_thres: float = 0.5,
                     containment_thres: float = 0.8,
                     include_conf_in_txt: bool = False,
                     verbose: bool = True):

        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Не удалось прочитать изображение {image_path}")
        
        h, w = image.shape[:2]
        
        final_boxes, stats = self._process_image_array(
            image, pyramid_levels, merge_iou_thres, 
            fuse_nms_thres, containment_thres, verbose
        )
        
        if txt_save_dir:
            image_name = Path(image_path).stem
            txt_path = Path(txt_save_dir) / f"{image_name}.txt"
            
            self.save_detections_to_txt(
                final_boxes, w, h, 
                str(txt_path), 
                include_conf=include_conf_in_txt
            )
            if verbose:
                print(f"  TXT сохранен: {txt_path}")
        
        result_image = image.copy()
        class_counts = {}
        
        for box in final_boxes:
            x1, y1, x2, y2, conf, cls = box[:6]
            source = box[6] if len(box) > 6 else 'unknown'
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cls_id = int(cls)
            
            class_counts[cls_id] = class_counts.get(cls_id, 0) + 1
            
            color = self.get_class_color(cls_id)
            thickness = 3 if source == 'global' else 2
            
            label = f"{self.class_names[cls_id]}: {conf:.2f} [{source}]"
            cv2.rectangle(result_image, (x1, y1), (x2, y2), color, thickness)
            cv2.putText(result_image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        legend_y = 30
        for cls_id, count in sorted(class_counts.items()):
            color = self.get_class_color(cls_id)
            legend_text = f"{self.class_names[cls_id]}: {count} objects"
            cv2.putText(result_image, legend_text, (10, legend_y), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            legend_y += 25
        
        global_count = sum(1 for b in final_boxes if len(b) > 6 and b[6] == 'global')
        tiled_count = len(final_boxes) - global_count
        stats_text = f"Global: {global_count} | Tiled: {tiled_count}"
        cv2.putText(result_image, stats_text, (10, legend_y + 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        if save_path:
            cv2.imwrite(save_path, result_image)
            if verbose:
                print(f"  Изображение сохранено: {save_path}")
        
        stats['image'] = image_path
        stats['class_distribution'] = class_counts
        
        return result_image, final_boxes, stats

    def process_frame(self, frame: np.ndarray, pyramid_levels: List[int] = None,
                     merge_iou_thres: float = 0.5, fuse_nms_thres: float = 0.5,
                     containment_thres: float = 0.8, verbose: bool = False):

        final_boxes, stats = self._process_image_array(
            frame, pyramid_levels, merge_iou_thres,
            fuse_nms_thres, containment_thres, verbose
        )
        
        result_frame = frame.copy()
        
        for box in final_boxes:
            x1, y1, x2, y2, conf, cls = box[:6]
            source = box[6] if len(box) > 6 else 'unknown'
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cls_id = int(cls)
            
            color = self.get_class_color(cls_id)
            thickness = 3 if source == 'global' else 1
            label = f"{self.class_names[cls_id]}:{conf:.2f}"
            cv2.rectangle(result_frame, (x1, y1), (x2, y2), color, thickness)
            cv2.putText(result_frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)
        
        return result_frame, final_boxes, stats

    def process_image_directory(self, input_dir: str, output_dir: str,
                               txt_output_dir: Optional[str] = None,
                               pyramid_levels: List[int] = None,
                               merge_iou_thres: float = 0.5,
                               fuse_nms_thres: float = 0.5,
                               containment_thres: float = 0.8,
                               include_conf_in_txt: bool = False,
                               image_extensions: List[str] = None,
                               verbose: bool = False):
        
        if image_extensions is None:
            image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp']
        
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        if txt_output_dir:
            Path(txt_output_dir).mkdir(parents=True, exist_ok=True)
            print(f"TXT файлы будут сохранены в: {txt_output_dir}")
        
        input_path = Path(input_dir)
        image_files = []
        
        for ext in image_extensions:
            image_files.extend(input_path.glob(f'*{ext}'))
            image_files.extend(input_path.glob(f'*{ext.upper()}'))
        
        image_files = sorted(list(set(image_files)))
        
        if not image_files:
            print(f"Не найдено изображений в папке {input_dir}")
            return
        
        print(f"\n{'='*60}")
        print(f"Пакетная обработка директории")
        print(f"Вход: {input_dir}")
        print(f"Выход (изображения): {output_dir}")
        print(f"Выход (TXT): {txt_output_dir or 'Не сохраняется'}")
        print(f"Изображений: {len(image_files)}")
        print(f"Уровни пирамиды: {pyramid_levels or [self.imgsz]}")
        print(f"{'='*60}\n")
        
        total_stats = {
            'total_images': len(image_files),
            'processed': 0,
            'failed': 0,
            'total_boxes': 0,
            'global_boxes': 0,
            'tiled_boxes': 0,
            'class_distribution': {},
            'per_image_stats': []
        }
        
        for img_path in tqdm(image_files, desc="Processing"):
            try:
                output_path = Path(output_dir) / img_path.name
                txt_path = None
                if txt_output_dir:
                    txt_path = Path(txt_output_dir) / f"{img_path.stem}.txt"
                
                result_img, boxes, stats = self.process_image(
                    str(img_path),
                    save_path=str(output_path),
                    txt_save_dir=str(txt_path.parent) if txt_path else None,
                    pyramid_levels=pyramid_levels,
                    merge_iou_thres=merge_iou_thres,
                    fuse_nms_thres=fuse_nms_thres,
                    containment_thres=containment_thres,
                    include_conf_in_txt=include_conf_in_txt,
                    verbose=verbose
                )
                
                total_stats['processed'] += 1
                total_stats['total_boxes'] += stats['final_boxes']
                total_stats['global_boxes'] += stats['global_in_final']
                total_stats['tiled_boxes'] += stats['tiled_in_final']
                
                for cls_id, count in stats['class_distribution'].items():
                    total_stats['class_distribution'][cls_id] = \
                        total_stats['class_distribution'].get(cls_id, 0) + count
                
                total_stats['per_image_stats'].append(stats)
                
                if verbose:
                    print(f"  ✓ {img_path.name}: {stats['final_boxes']} объектов "
                          f"(G:{stats['global_in_final']}, T:{stats['tiled_in_final']})\n")
                    
            except Exception as e:
                total_stats['failed'] += 1
                print(f"  ✗ Ошибка {img_path.name}: {e}")
        
        print(f"\n{'='*60}")
        print("ИТОГОВАЯ СТАТИСТИКА")
        print(f"{'='*60}")
        print(f"Всего изображений: {total_stats['total_images']}")
        print(f"Успешно: {total_stats['processed']}")
        print(f"Ошибок: {total_stats['failed']}")
        print(f"Всего объектов: {total_stats['total_boxes']}")
        print(f"  Из глобального (4K): {total_stats['global_boxes']}")
        print(f"  Из тайлов: {total_stats['tiled_boxes']}")
        
        if total_stats['class_distribution']:
            print(f"\nПо классам:")
            for cls_id, count in sorted(total_stats['class_distribution'].items()):
                print(f"  {self.class_names[cls_id]}: {count}")
        
        print(f"{'='*60}\n")
        
        stats_path = Path(output_dir) / 'detection_stats.json'
        with open(stats_path, 'w', encoding='utf-8') as f:
            json.dump(total_stats, f, ensure_ascii=False, indent=2)
        print(f"Статистика: {stats_path}")
        
        return total_stats

    def process_video(self, video_path: str, output_path: str,
                     pyramid_levels: List[int] = None,
                     merge_iou_thres: float = 0.5,
                     fuse_nms_thres: float = 0.5,
                     containment_thres: float = 0.8):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Не удалось открыть видео {video_path}")
            
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
        
        frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            if frame_count % 10 == 0:
                print(f"Кадр {frame_count}...")
            
            result_frame, boxes, stats = self.process_frame(
                frame,
                pyramid_levels=pyramid_levels,
                merge_iou_thres=merge_iou_thres,
                fuse_nms_thres=fuse_nms_thres,
                containment_thres=containment_thres,
                verbose=False
            )
                
            out.write(result_frame)
            
        cap.release()
        out.release()
        print(f"Видео сохранено: {output_path}")

Запуск для директории с сохранением предсказаний

In [ ]:
MODEL_PATH = "/wrk/YOLO/runs/new_data/detect/yolo26_train_20260421_140047/weights/best.pt"
    
pipeline = SmartYOLOInference(model_path=MODEL_PATH, imgsz=640, conf_thres=0.3)

# --- Обработка директории с сохранением TXT ---
pipeline.process_image_directory(
    input_dir="/wrk/pyramid_test/data/test/images",
    output_dir="/wrk/pyramid_test/data/test/results",
    txt_output_dir="/wrk/pyramid_test/data/test/preds",
    pyramid_levels=[640], # Сколько уровней + какого разрешения кропы
    merge_iou_thres=0.5,
    fuse_nms_thres=0.5,
    containment_thres=0.7,
    include_conf_in_txt=True,  # True если нужен confidence в TXT
    verbose=False
)

Запуск для одного изображения

In [ ]:
MODEL_PATH = "/wrk/YOLO/runs/new_data/detect/yolo26_train_20260421_140047/weights/best.pt"
    
pipeline = SmartYOLOInference(model_path=MODEL_PATH, imgsz=640, conf_thres=0.3)

# --- Обработка одного изображения с TXT ---
result, boxes, stats = pipeline.process_image(
    "input_4k.jpg",
    save_path="output_4k.jpg",
    txt_save_dir="./labels",  # ← Папка для TXT
    pyramid_levels=[320, 640, 1280],
    include_conf_in_txt=False,
    verbose=True
)

Запуск для видео

In [ ]:
MODEL_PATH = "/wrk/YOLO/runs/new_data/detect/yolo26_train_20260421_140047/weights/best.pt"
    
pipeline = SmartYOLOInference(model_path=MODEL_PATH, imgsz=640, conf_thres=0.3)

# --- Обработка видео ---
pipeline.process_video(
    video_path=".mp4",
    output_path=".mp4",
    pyramid_levels=[640],
    merge_iou_thres=0.5,
    fuse_nms_thres=0.5,
    containment_thres=0.7
)

### Metrics

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# === КОНФИГУРАЦИЯ ===
IMG_DIR = "/wrk/pyramid_test/data/test/images"
PRED_DIR = "/wrk/pyramid_test/data/test/preds"
GT_DIR = "/wrk/pyramid_test/data/test/labels"
CONF_THRESHOLD = 0.30   # Порог уверенности для Precision/Recall/F1/CM
IOU_THRESHOLD = 0.5     # IoU для совпадения и mAP@50

def load_image_sizes(img_dir):
    sizes = {}
    for path in Path(img_dir).iterdir():
        if path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'):
            try:
                with Image.open(path) as img:
                    sizes[path.stem] = img.size
            except Exception:
                pass
    return sizes

def parse_yolo_to_coco_format(txt_path, img_w, img_h, img_id, ann_start_id, is_prediction=False):
    annotations = []
    current_id = ann_start_id
    
    if not os.path.exists(txt_path):
        return annotations, current_id
        
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            
            try:
                cls = int(float(parts[0]))
                cx, cy, bw, bh = map(float, parts[1:5])
                
                x = (cx - bw / 2) * img_w
                y = (cy - bh / 2) * img_h
                width = bw * img_w
                height = bh * img_h
                
                if width <= 0 or height <= 0:
                    continue
                
                ann = {
                    "id": current_id,
                    "image_id": img_id,
                    "category_id": cls,
                    "bbox": [x, y, width, height],
                    "area": width * height,
                    "iscrowd": 0
                }
                
                if is_prediction:
                    conf = float(parts[5]) if len(parts) > 5 else 1.0
                    ann["score"] = conf
                
                annotations.append(ann)
                current_id += 1
                
            except ValueError:
                continue
                
    return annotations, current_id

def compute_map50_coco(pred_dir, gt_dir, img_dir, img_sizes):
    

    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    if not img_files:
        raise FileNotFoundError("Не найдено изображений в IMG_DIR")

    gt_json = {"images": [], "annotations": [], "categories": []}
    pred_annotations = []
    
    all_classes = set()
    img_id_counter = 1
    ann_gt_id = 1
    ann_pred_id = 1
    
    print(f"   📂 Подготовка данных для COCO...")

    for f in img_files:
        stem = Path(f).stem
        w, h = img_sizes.get(stem, (640, 640))
        current_img_id = img_id_counter
        img_id_counter += 1
        
        img_info = {"id": current_img_id, "file_name": f, "width": int(w), "height": int(h)}
        gt_json["images"].append(img_info)
        
        gt_path = os.path.join(gt_dir, stem + ".txt")
        gts, ann_gt_id = parse_yolo_to_coco_format(gt_path, w, h, current_img_id, ann_gt_id, is_prediction=False)
        gt_json["annotations"].extend(gts)
        for g in gts: all_classes.add(g['category_id'])
            
        pred_path = os.path.join(pred_dir, stem + ".txt")
        preds, ann_pred_id = parse_yolo_to_coco_format(pred_path, w, h, current_img_id, ann_pred_id, is_prediction=True)
        pred_annotations.extend(preds)
        for p in preds: all_classes.add(p['category_id'])

    if not all_classes:
        print("⚠️ Не найдено классов. mAP = 0")
        return 0.0

    categories = [{"id": c, "name": f"class_{c}", "supercategory": "none"} for c in sorted(all_classes)]
    gt_json["categories"] = categories

    temp_gt = "_temp_gt.json"
    temp_pred = "_temp_pred.json"

    with open(temp_gt, "w") as f:
        json.dump(gt_json, f)
    with open(temp_pred, "w") as f:
        json.dump(pred_annotations, f)

    try:
        coco_gt = COCO(temp_gt)
        coco_pred = coco_gt.loadRes(temp_pred)
        
        eval = COCOeval(coco_gt, coco_pred, "bbox")
        eval.params.iouThrs = np.array([IOU_THRESHOLD])
        eval.evaluate()
        eval.accumulate()
        eval.summarize()
        
        ap50 = eval.stats[0] if len(eval.stats) > 0 else 0.0
        return ap50
        
    except Exception as e:
        print(f"❌ Ошибка pycocotools: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_gt): os.remove(temp_gt)
        if os.path.exists(temp_pred): os.remove(temp_pred)

def match_predictions_and_gt_simple(preds, gts, iou_thresh, conf_thresh):
    valid_preds = [p for p in preds if p['conf'] >= conf_thresh]
    valid_preds.sort(key=lambda x: x['conf'], reverse=True)
    
    matched_gts = set()
    tp_per_cls = {}
    fp_per_cls = {}
    matches = [] 
    
    for pred in valid_preds:
        best_iou = 0
        best_gt_idx = -1
        for i, gt in enumerate(gts):
            if i in matched_gts: continue
            
            x1 = max(pred['x1'], gt['x1'])
            y1 = max(pred['y1'], gt['y1'])
            x2 = min(pred['x2'], gt['x2'])
            y2 = min(pred['y2'], gt['y2'])
            
            inter = max(0, x2 - x1) * max(0, y2 - y1)
            area_p = (pred['x2'] - pred['x1']) * (pred['y2'] - pred['y1'])
            area_g = (gt['x2'] - gt['x1']) * (gt['y2'] - gt['y1'])
            union = area_p + area_g - inter
            iou = inter / union if union > 0 else 0
            
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i
                
        if best_iou >= iou_thresh and best_gt_idx != -1:
            matched_gts.add(best_gt_idx)
            p_cls, g_cls = pred['cls'], gts[best_gt_idx]['cls']
            tp_per_cls[p_cls] = tp_per_cls.get(p_cls, 0) + 1
            matches.append((p_cls, g_cls))
        else:
            fp_per_cls[pred['cls']] = fp_per_cls.get(pred['cls'], 0) + 1
            
    fn_per_cls = {}
    for i, gt in enumerate(gts):
        if i not in matched_gts:
            fn_per_cls[gt['cls']] = fn_per_cls.get(gt['cls'], 0) + 1
            
    return tp_per_cls, fp_per_cls, fn_per_cls, matches

def plot_confusion_matrix(cm_data, classes, save_path="confusion_matrix.png"):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[str(c) for c in classes],
                yticklabels=[str(c) for c in classes])
    plt.title('Confusion Matrix (Predicted vs Ground Truth)')
    plt.xlabel('Ground Truth Class')
    plt.ylabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"🖼️ Матрица сохранена в {save_path}")


if __name__ == "__main__":
    print("📏 Загрузка размеров изображений...")
    img_sizes = load_image_sizes(IMG_DIR)
    
    all_tp, all_fp, all_fn, all_matches = {}, {}, {}, []
    img_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    
    print(f"🔍 Обработка {len(img_files)} изображений для метрик P/R/F1...")
    
    def parse_for_metrics(txt_path, w, h):
        boxes = []
        if not os.path.exists(txt_path): return boxes
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                try:
                    cls = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    conf = float(parts[5]) if len(parts) > 5 else 1.0
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    x2 = (cx + bw/2) * w
                    y2 = (cy + bh/2) * h
                    boxes.append({'cls': cls, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'conf': conf})
                except ValueError: continue
        return boxes

    for f in img_files:
        stem = Path(f).stem
        w, h = img_sizes.get(stem, (640, 640))
        
        gts = parse_for_metrics(os.path.join(GT_DIR, stem + ".txt"), w, h)
        preds = parse_for_metrics(os.path.join(PRED_DIR, stem + ".txt"), w, h)
        
        tp, fp, fn, matches = match_predictions_and_gt_simple(preds, gts, IOU_THRESHOLD, CONF_THRESHOLD)
        all_matches.extend(matches)
        
        for k in set(list(tp.keys()) + list(fp.keys()) + list(fn.keys())):
            all_tp[k] = all_tp.get(k, 0) + tp.get(k, 0)
            all_fp[k] = all_fp.get(k, 0) + fp.get(k, 0)
            all_fn[k] = all_fn.get(k, 0) + fn.get(k, 0)
            
    classes = sorted(set(list(all_tp.keys()) + list(all_fp.keys()) + list(all_fn.keys())))
    
    if not classes:
        print("⚠️ Нет классов.")
        exit()

    tp_arr = np.array([all_tp.get(c, 0) for c in classes])
    fp_arr = np.array([all_fp.get(c, 0) for c in classes])
    fn_arr = np.array([all_fn.get(c, 0) for c in classes])
    
    prec = tp_arr / (tp_arr + fp_arr + 1e-8)
    rec = tp_arr / (tp_arr + fn_arr + 1e-8)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    
    print("\n📊 По-классовые метрики:")
    print(f"{'Class':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec':<6} | {'Recall':<7} | {'F1':<6}")
    print("-" * 65)
    for c in classes:
        idx = classes.index(c)
        print(f"{c:<8} | {tp_arr[idx]:<5} | {fp_arr[idx]:<5} | {fn_arr[idx]:<5} | {prec[idx]:<6.3f} | {rec[idx]:<7.3f} | {f1[idx]:<6.3f}")
        
    mean_p = np.mean(prec)
    mean_r = np.mean(rec)
    mean_f1 = np.mean(f1)
    print(f"\n📈 Средние (Macro): Prec={mean_p:.3f}, Recall={mean_r:.3f}, F1={mean_f1:.3f}")

    if all_matches:
        cm_size = len(classes)
        cm_data = np.zeros((cm_size, cm_size), dtype=int)
        for p_cls, g_cls in all_matches:
            if p_cls in classes and g_cls in classes:
                r = classes.index(p_cls)
                c = classes.index(g_cls)
                cm_data[r, c] += 1
        plot_confusion_matrix(cm_data, classes)
        
        print("\n🔲 Текстовая Confusion Matrix:")
        print("   | " + " | ".join(f"{c:^6}" for c in classes) + " | FP")
        print("-" * (10 + 7*(len(classes)+1)))
        for i, c in enumerate(classes):
            row = cm_data[i].tolist() + [all_fp.get(c, 0)]
            print(f"{c:<3}| " + " | ".join(f"{v:^6}" for v in row))
        fn_row = [all_fn.get(c, 0) for c in classes]
        print(f"FN | " + " | ".join(f"{v:^6}" for v in fn_row) + " | -")

    print("\n📐 Расчёт mAP@50...")
    try:
        map50 = compute_map50_coco(PRED_DIR, GT_DIR, IMG_DIR, img_sizes)
        print(f"✅ mAP@50 = {map50:.4f}")
    except Exception as e:
        print(f"❌ Ошибка mAP: {e}")